# <div style="background-color: #7851A9; padding: 12px; font-size: 18px; color: #FFFFFF; border-left: 8px solid #C9A0DC; width: 100%;"><b>🏃Competiton : Predict Calorie Expenditure</b></div>

## LGBM : https://www.kaggle.com/code/leonchani/01-only-lgbm-calorie-expenditure/notebook
## XGB  : https://www.kaggle.com/code/leonchani/02-only-xgb-calorie-expenditure/notebook
## CBT  : https://www.kaggle.com/code/leonchani/03-only-cbt-calorie-expenditure/notebook
## MLP  : https://www.kaggle.com/code/leonchani/04-only-mlp-calorie-expenditure/notebook
## RF   : https://www.kaggle.com/code/leonchani/05-only-randomforest-calorie-expenditure/notebook

# <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">Importing Libraries</span></b>

In [1]:
# Table manipulation, calculating
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 100) # increase the maximum number of columns

# Saving model
import xgboost as xgb
import joblib

# Ignore all warnings
import warnings
warnings.simplefilter("ignore")

# <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">Importing Datasets</span>

In [2]:
df_train = pd.read_csv('/kaggle/input/playground-series-s5e5/train.csv')
df_test = pd.read_csv('/kaggle/input/playground-series-s5e5/test.csv')

In [3]:
df_train

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0
...,...,...,...,...,...,...,...,...,...
749995,749995,male,28,193.0,97.0,30.0,114.0,40.9,230.0
749996,749996,female,64,165.0,63.0,18.0,92.0,40.5,96.0
749997,749997,male,60,162.0,67.0,29.0,113.0,40.9,221.0
749998,749998,male,45,182.0,91.0,17.0,102.0,40.3,109.0


In [4]:
df_test

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,750000,male,45,177.0,81.0,7.0,87.0,39.8
1,750001,male,26,200.0,97.0,20.0,101.0,40.5
2,750002,female,29,188.0,85.0,16.0,102.0,40.4
3,750003,female,39,172.0,73.0,20.0,107.0,40.6
4,750004,female,30,173.0,67.0,16.0,94.0,40.5
...,...,...,...,...,...,...,...,...
249995,999995,female,56,159.0,62.0,6.0,85.0,39.4
249996,999996,male,32,202.0,101.0,3.0,84.0,38.4
249997,999997,female,31,164.0,64.0,14.0,98.0,40.1
249998,999998,female,62,158.0,61.0,25.0,106.0,40.7


# <div style="background-color: #7851A9; padding: 12px; font-size: 18px; color: #FFFFFF; border-left: 8px solid #C9A0DC"><b>📊EDA</b>

## Detail: https://www.kaggle.com/code/leonchani/eda-inspection-calorie-expenditure

# <div style="background-color: #7851A9; padding: 12px; font-size: 18px; color: #FFFFFF; border-left: 8px solid #C9A0DC"><b>🤔Feature Engineering</b>


In [5]:
mapping = {'male': 1.5, 'female': 1.0}
df_train['Sex'] = df_train['Sex'].replace(mapping)
df_test['Sex']  = df_test['Sex'].replace(mapping)

In [6]:
def standardize_dataframe(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Standardizes the columns of the specified DataFrame and returns the updated original DataFrame.
    
    Args:
        df (pd.DataFrame): The DataFrame to standardize.
        cols (list[str]): A list of column names to standardize.
    
    Returns:
        pd.DataFrame: The DataFrame with the specified columns standardized (modifies the original DataFrame).

    """
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    scaler.fit(df[cols])
    scaled_values = scaler.transform(df[cols])
    df[cols] = scaled_values

    return df

In [7]:
# columns_to_standardize = ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads']
columns_to_drop = ['id', 'Calories']
columns_to_standardize = df_train.copy().drop(columns=columns_to_drop).columns

standardize_dataframe(df_train, columns_to_standardize)
standardize_dataframe(df_test, columns_to_standardize)

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,750000,1.002251,0.233733,0.177381,0.418634,-1.007942,-0.897244,-0.303287
1,750001,1.002251,-1.018101,1.971171,1.563168,0.549109,0.584215,0.595940
2,750002,-0.997755,-0.820443,1.035280,0.704768,0.070016,0.690034,0.467479
3,750003,-0.997755,-0.161583,-0.212574,-0.153633,0.549109,1.219126,0.724402
4,750004,-0.997755,-0.754557,-0.134583,-0.582834,0.070016,-0.156514,0.595940
...,...,...,...,...,...,...,...,...
249995,999995,-0.997755,0.958478,-1.226455,-0.940500,-1.127716,-1.108881,-0.817131
249996,999996,1.002251,-0.622785,2.127152,1.849302,-1.487035,-1.214700,-2.101742
249997,999997,-0.997755,-0.688671,-0.836501,-0.797434,-0.169530,0.266760,0.082096
249998,999998,-0.997755,1.353794,-1.304446,-1.012034,1.147974,1.113308,0.852863


## <div style="background-color: #7851A9; padding: 12px; font-size: 18px; color: #FFFFFF; border-left: 8px solid #C9A0DC"><b>🔍Loading Models</b>

In [8]:
# Loading Models
light_gbm     = joblib.load('/kaggle/input/model_for_ensemble/scikitlearn/default/1/LightGBM.joblib')
xgboost       = joblib.load('/kaggle/input/model_for_ensemble/scikitlearn/default/1/XGBoost.joblib')
catboost      = joblib.load('/kaggle/input/model_for_ensemble/scikitlearn/default/1/CBT.joblib')
mlp           = joblib.load('/kaggle/input/model_for_ensemble/scikitlearn/default/1/MLP.joblib')
random_forest = joblib.load('/kaggle/input/model_for_ensemble/scikitlearn/default/1/RandomForest.joblib')

## <div style="background-color: #7851A9; padding: 12px; font-size: 18px; color: #FFFFFF; border-left: 8px solid #C9A0DC"><b>🔍Prediction</b>

In [9]:
test_id = df_test["id"]
test = df_test.drop(columns=['id'])

lgbm_submit_score = []
xgb_submit_score  = []
cbt_submit_score  = []
mlp_submit_score  = []
rf_submit_score   = []

## <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">LightGBM</span>

In [10]:
for fold_, model in enumerate(light_gbm):
    pred_ = model.predict(test)
    lgbm_submit_score.append(pred_)

lgbm_pred = np.mean(lgbm_submit_score, axis=0)
lgbm_pred = np.expm1(lgbm_pred)

## <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">XGBoost</span>

In [11]:
# for fold_, model in enumerate(xgboost):
#     dtest = xgb.DMatrix(test)
#     pred_ = model.predict(dtest)
#     xgb_submit_score.append(pred_)

# xgb_pred = np.mean(xgb_submit_score, axis=0)

## <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">CatBoost</span>

In [12]:
for fold_, model in enumerate(catboost):
    pred_ = model.predict(test)
    cbt_submit_score.append(pred_)

cbt_pred = np.mean(cbt_submit_score, axis=0)
cbt_pred = np.expm1(cbt_pred)

## <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">MLP</span>

In [13]:
# for fold_, model in enumerate(mlp):
#     pred_ = model.predict(test)
#     mlp_submit_score.append(pred_)

# mlp_pred = np.mean(mlp_submit_score, axis=0)
# mlp_pred = np.expm1(mlp_pred)

## <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">RandomForest</span>

In [14]:
# for fold_, model in enumerate(random_forest):
#     pred_ = model.predict(test)
#     rf_submit_score.append(pred_)

# rf_pred = np.mean(rf_submit_score, axis=0)
# rf_pred = np.expm1(rf_pred)

# <b><span style="color: #FFFFFF; background-color: #7851A9; padding: 20px; font-size: 18px; border-left: 8px solid #C9A0DC">Merge Predict Score
</span>

In [15]:
# Merge Predict Score
df_lgbm_score = pd.DataFrame(lgbm_pred, columns=['lgbm_score']).reset_index()
df_cbt_score  = pd.DataFrame(cbt_pred, columns=['cbt_score']).reset_index()
# df_mlp_score  = pd.DataFrame(mlp_pred, columns=['mlp_score']).reset_index()
# df_rf_score   = pd.DataFrame(rf_pred, columns=['rf_score']).reset_index()

In [16]:
df_score = pd.merge(df_lgbm_score, df_cbt_score, how = 'inner', on = 'index')
# df_score = pd.merge(df_score, df_mlp_score, how = 'inner', on = 'index')
# df_score = pd.merge(df_score, df_rf_score, how = 'inner', on = 'index')
df_score

,index,lgbm_score,cbt_score
0,0,26.984701,27.216815
1,1,109.189395,108.144324
2,2,86.557387,86.865674
3,3,123.906721,125.907476
4,4,76.042791,76.046148
...,...,...,...
249995,249995,26.070886,26.191374
249996,249996,9.314345,9.475181
249997,249997,72.519569,73.562641
249998,249998,168.585550,168.790784


In [17]:
# df_score['pred'] = (df_score['lgbm_score'] + df_score['cbt_score'] + df_score['mlp_score'] + df_score['rf_score']) / 4
df_score['pred'] = (df_score['lgbm_score'] + df_score['cbt_score']) / 2
df_score

,index,lgbm_score,cbt_score,pred
0,0,26.984701,27.216815,27.100758
1,1,109.189395,108.144324,108.666859
2,2,86.557387,86.865674,86.711531
3,3,123.906721,125.907476,124.907099
4,4,76.042791,76.046148,76.044470
...,...,...,...,...
249995,249995,26.070886,26.191374,26.131130
249996,249996,9.314345,9.475181,9.394763
249997,249997,72.519569,73.562641,73.041105
249998,249998,168.585550,168.790784,168.688167


## <div style="background-color: #7851A9; padding: 12px; font-size: 18px; color: #FFFFFF; border-left: 8px solid #C9A0DC"><b>📜Submission</b>

In [18]:
submission = pd.DataFrame({
    'id': test_id,
    'Calories': df_score['pred']
})

submission['Calories'] = submission['Calories'].apply(lambda x: 0 if x < 0 else x)

# Save
submission.to_csv('submission.csv', index=False)

submission

,id,Calories
0,750000,27.100758
1,750001,108.666859
2,750002,86.711531
3,750003,124.907099
4,750004,76.044470
...,...,...
249995,999995,26.131130
249996,999996,9.394763
249997,999997,73.041105
249998,999998,168.688167


# <div style="background-color: #7851A9; padding: 12px; font-size: 18px; color: #FFFFFF; border-left: 8px solid #C9A0DC"><b>Check Libiraries Version</b></div>

In [19]:
!pip install watermark

In [20]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pytensor,aeppl,xarray

Last updated: Thu May 22 2025

Python implementation: CPython
Python version       : 3.11.11
IPython version      : 7.34.0

pytensor: 2.30.2
aeppl   : not installed
xarray  : 2025.1.2

numpy  : 1.26.4
joblib : 1.5.0
pandas : 2.2.3
xgboost: 2.0.3

Watermark: 2.5.0

